# Test A: test_data.py

Ensures that the raw hospital dataset loads correctly, produces a non-empty DataFrame, contains required predictor variables, and enforces valid ESI target scores.

In [34]:
!mkdir -p src scripts tests data models figs reports

In [35]:
%%writefile config.yaml
seed: 42

data:
  raw_path: "data/yaleemmlc_admissionprediction_triage.csv"
  target: "esi"

model:
  model_type: "random_forest"
  hyperparameters:
    n_estimators: 200
    class_weight: "balanced"
    random_state: 42
    n_jobs: -1

paths:
  figures: "figs"
  models: "models"
  reports: "reports"

Overwriting config.yaml


In [36]:
%%writefile src/__init__.py
# Empty file to make src a Python package

Overwriting src/__init__.py


In [37]:
%%writefile src/utils.py
"""
Shared utility helper functions for the Emergency Department triage pipeline.
"""

import argparse
import os
import yaml


def parse_args():
    parser = argparse.ArgumentParser(
        description="Train the Emergency Department triage prediction model."
    )
    parser.add_argument(
        "--config",
        type=str,
        default="config.yaml",
        help="Path to the YAML configuration file."
    )
    return parser.parse_args()


def load_config(config_path: str) -> dict:
    if not os.path.exists(config_path):
        raise FileNotFoundError(
            f"Configuration file not found: '{os.path.abspath(config_path)}'"
        )

    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def create_folder(folder_path: str):
    os.makedirs(folder_path, exist_ok=True)


def format_time(seconds: float) -> str:
    if seconds >= 3600:
        return f"{seconds / 3600:.2f} hours"
    if seconds >= 60:
        return f"{seconds / 60:.2f} minutes"
    if seconds >= 1:
        return f"{seconds:.3f} seconds"
    return f"{seconds * 1000:.3f} milliseconds"

Overwriting src/utils.py


In [38]:
%%writefile src/data.py
"""
Data ingestion and sanitation module for ED triage data.
"""

import pandas as pd
import numpy as np

TARGET = "esi"

VITAL_SIGNS = [
    "triage_vital_dbp",
    "triage_vital_hr",
    "triage_vital_o2",
    "triage_vital_rr",
    "triage_vital_sbp",
    "triage_vital_temp",
    "triage_glucose",
]


def load_data(file_path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(file_path)
        print(f"✓ Loaded dataset from {file_path} (Shape: {df.shape})")
        return df
    except Exception as e:
        raise FileNotFoundError(f"Failed to load dataset at '{file_path}': {str(e)}")


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    if TARGET not in df_clean.columns:
        raise KeyError(f"Target column '{TARGET}' not found in dataset.")

    df_clean = df_clean.dropna(subset=[TARGET])
    df_clean[TARGET] = pd.to_numeric(df_clean[TARGET], errors="coerce")
    df_clean = df_clean[df_clean[TARGET].isin([1.0, 2.0, 3.0, 4.0, 5.0])]
    df_clean[TARGET] = df_clean[TARGET].astype(int)

    for col in VITAL_SIGNS:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    return df_clean

Overwriting src/data.py


In [39]:
%%writefile src/features.py
"""
Feature engineering, selection, and preprocessing module.
"""

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

TARGET = "esi"

VITAL_SIGNS = [
    "triage_vital_dbp",
    "triage_vital_hr",
    "triage_vital_o2",
    "triage_vital_rr",
    "triage_vital_sbp",
    "triage_vital_temp",
    "triage_glucose",
]

DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "insurance_type"]
ADMIN = ["Unnamed: 0", "dep_name", "patient_id", "encounter_id", "visit_id", "disposition_id"]
LEAKAGE = [
    "n_edvisits", "n_admissions", "admitted_to_hospital", "admission_type",
    "admission_source", "discharge_disposition", "discharge_destination",
    "length_of_stay", "admission_to_icu_flag", "icu_stay_days",
    "mortality_flag", "disposition_dt", "discharge_dt",
]


def add_clinical_features(df: pd.DataFrame) -> pd.DataFrame:
    df_eng = df.copy()

    if "triage_vital_hr" in df_eng.columns and "triage_vital_sbp" in df_eng.columns:
        df_eng["eng_shock_index"] = (
            df_eng["triage_vital_hr"] / df_eng["triage_vital_sbp"].replace(0, np.nan)
        )

    if "triage_vital_hr" in df_eng.columns:
        df_eng["eng_bradycardia"] = (df_eng["triage_vital_hr"] < 60).astype(int)

    if "triage_glucose" in df_eng.columns:
        df_eng["eng_hyperglycemia"] = (df_eng["triage_glucose"] > 180).astype(int)

    if "triage_vital_temp" in df_eng.columns:
        df_eng["eng_hypothermia"] = (df_eng["triage_vital_temp"] < 35).astype(int)

    if "triage_vital_rr" in df_eng.columns:
        df_eng["eng_respiratory_distress"] = (df_eng["triage_vital_rr"] > 20).astype(int)

    if "triage_vital_o2" in df_eng.columns and "triage_vital_rr" in df_eng.columns:
        df_eng["eng_o2_rr_ratio"] = (
            df_eng["triage_vital_o2"] / df_eng["triage_vital_rr"].replace(0, np.nan)
        )

    return df_eng


def select_features(df: pd.DataFrame) -> pd.DataFrame:
    exclude_cols = set(DEMOGRAPHICS + ADMIN + LEAKAGE + [TARGET])
    chief_complaints = [col for col in df.columns if col.startswith("cc_")]
    engineered_cols = [col for col in df.columns if col.startswith("eng_")]

    candidate_features = set(VITAL_SIGNS + chief_complaints + engineered_cols)

    final_features = [
        col for col in candidate_features
        if col in df.columns and col not in exclude_cols
    ]

    return df[final_features].copy()


def impute_features(X_train: pd.DataFrame, X_test: pd.DataFrame):
    imputer = SimpleImputer(strategy="median")
    X_train_array = imputer.fit_transform(X_train)
    X_test_array = imputer.transform(X_test)

    X_train_imp = pd.DataFrame(X_train_array, columns=X_train.columns, index=X_train.index)
    X_test_imp = pd.DataFrame(X_test_array, columns=X_test.columns, index=X_test.index)

    return X_train_imp, X_test_imp, imputer

Overwriting src/features.py


In [40]:
%%writefile tests/test_data.py
"""
Sanity tests for data loading and schema validation (Task 4a).
"""

import sys
from pathlib import Path

# Add project root directory to Python path
ROOT_DIR = Path(__file__).resolve().parent.parent
sys.path.append(str(ROOT_DIR))

import pandas as pd
import pytest
from src.data import load_data, clean_data, TARGET, VITAL_SIGNS
from src.utils import load_config


def test_data_loading_and_schema():
    config = load_config("config.yaml")
    raw_path = config["data"]["raw_path"]

    raw_df = load_data(raw_path)
    assert isinstance(raw_df, pd.DataFrame), "load_data() should return a DataFrame"
    assert not raw_df.empty, "Raw loaded DataFrame is empty!"

    cleaned_df = clean_data(raw_df)
    assert not cleaned_df.empty, "Cleaned DataFrame should not be empty!"

    assert TARGET in cleaned_df.columns, f"Target column '{TARGET}' missing!"

    unique_targets = set(cleaned_df[TARGET].unique())
    assert unique_targets.issubset({1, 2, 3, 4, 5}), (
        f"Corrupt ESI target values detected: {unique_targets}"
    )

    for vital_col in VITAL_SIGNS:
        assert vital_col in cleaned_df.columns, (
            f"Expected vital sign '{vital_col}' missing from schema!"
        )

Overwriting tests/test_data.py


In [41]:
!pip install pytest -q
!PYTHONPATH=. pytest tests/test_data.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collected 1 item                                                               

tests/test_data.py .                                                     [100%]

============================== 1 passed in 2.87s ===============================


# Test B: test_train.py

Runs an end-to-end execution of the data pipeline on a small 50-row subset to verify the model builds, trains, and evaluates without error.

In [42]:
%%writefile src/model.py
"""
Model building, training, evaluation, and persistence module.
"""

import time
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def build_model(config: dict):
    model_type = config.get("model_type", "random_forest").lower()
    params = config.get("hyperparameters", {})

    if model_type == "random_forest":
        return RandomForestClassifier(**params)
    elif model_type == "logistic_regression":
        return LogisticRegression(**params)
    elif model_type == "gradient_boosting":
        return GradientBoostingClassifier(**params)
    else:
        raise ValueError(f"Unsupported model_type: '{model_type}'")


def train_model(model, X_train, y_train):
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    end_time = time.perf_counter()

    training_time = end_time - start_time
    return model, training_time


def predict(model, X_test):
    return model.predict(X_test)


def evaluate_model(model, X_test, y_test) -> dict:
    start_time = time.perf_counter()
    y_pred = model.predict(X_test)
    end_time = time.perf_counter()

    total_inference_time = end_time - start_time
    latency_per_sample_ms = (total_inference_time / len(X_test)) * 1000

    accuracy = accuracy_score(y_test, y_pred)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )

    metrics = {
        "Accuracy": float(accuracy),
        "Weighted F1": float(weighted_f1),
        "Macro Precision": float(macro_precision),
        "Macro Recall": float(macro_recall),
        "Macro F1": float(macro_f1),
        "Inference Time Total (s)": float(total_inference_time),
        "Latency Per Patient (ms)": float(latency_per_sample_ms),
    }

    classes = sorted(list(np.unique(np.concatenate([y_test, y_pred]))))
    per_class_precision, per_class_recall, per_class_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, labels=classes, average=None, zero_division=0
    )

    for idx, esi_level in enumerate(classes):
        level_int = int(esi_level)
        metrics[f"ESI-{level_int} Recall"] = float(per_class_recall[idx])

    return metrics


def save_model(model, file_path: str):
    joblib.dump(model, file_path)


def load_model(file_path: str):
    return joblib.load(file_path)

Overwriting src/model.py


In [43]:
%%writefile tests/test_train.py
"""
Sanity smoke test for the training pipeline (Task 4b).
"""

import sys
from pathlib import Path

# Add project root directory to Python path
ROOT_DIR = Path(__file__).resolve().parent.parent
sys.path.append(str(ROOT_DIR))

import pandas as pd
from sklearn.model_selection import train_test_split

from src.data import load_data, clean_data
from src.features import add_clinical_features, select_features, impute_features
from src.model import build_model, train_model, evaluate_model
from src.utils import load_config


def test_pipeline_smoke_test():
    """
    Test 4(b): 50-Row Training Pipeline Smoke Test.

    Loads a tiny 50-row sample of the dataset and runs it through:
      Data cleaning -> Feature engineering -> Imputation -> Training -> Evaluation.

    Verifies that the whole pipeline completes without raising exceptions.
    """
    # 1. Load config
    config = load_config("config.yaml")
    raw_path = config["data"]["raw_path"]

    # 2. Ingest data and slice a tiny 50-row subset
    raw_df = load_data(raw_path)
    sample_df = raw_df.head(50).copy()
    assert len(sample_df) == 50, "Sample dataframe should contain exactly 50 rows!"

    # 3. Data cleaning
    cleaned_df = clean_data(sample_df)
    assert not cleaned_df.empty, "Cleaned sample dataframe should not be empty!"

    # 4. Feature engineering & selection
    engineered_df = add_clinical_features(cleaned_df)
    target_col = config["data"]["target"]
    y = engineered_df[target_col].astype(int)
    X = select_features(engineered_df)

    assert X.shape[0] == len(y), "Predictors (X) and Target (y) must have matching row counts!"

    # 5. Imputation & Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    X_train_imp, X_test_imp, _ = impute_features(X_train, X_test)

    # 6. Model Construction & Training
    model = build_model(config["model"])
    trained_model, train_time = train_model(model, X_train_imp, y_train)

    assert trained_model is not None, "Trained model object should not be None!"
    assert train_time >= 0, "Training time calculation should be non-negative!"

    # 7. Model Evaluation
    metrics = evaluate_model(trained_model, X_test_imp, y_test)

    assert isinstance(metrics, dict), "evaluate_model() must return a dictionary of metrics!"
    assert "Accuracy" in metrics, "Metrics dictionary must contain 'Accuracy' key!"

Overwriting tests/test_train.py


In [44]:
!PYTHONPATH=. pytest tests/

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collected 2 items                                                              

tests/test_data.py .                                                     [ 50%]
tests/test_train.py .                                                    [100%]

============================== 2 passed in 15.65s ==============================
